In [ ]:
pip install vmdpy holidays tqdm

In [ ]:
import random, os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from datetime import timedelta
import holidays
from vmdpy import VMD
from tqdm import tqdm
from scipy.stats import zscore
 
DATA_CSV      = "data_test1.csv"
TEST_RATIO    = 0.20
PAIS          = "DE"
DAYS_PREDICT  = 8
SEQ_LENGTH    = 48
PRED_LENGTH   = 24
BATCH_SIZE    = 128
EPOCHS        = 100
PATIENCE_ES   = 20
EPSILON_MAPE  = 5
CORR_THRESHOLD = 0.2
 

In [ ]:
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    os.environ["PYTHONHASHSEED"] = str(seed)
 
set_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Dispositivo: {device}")
 
df_raw = pd.read_csv(DATA_CSV)
 
# Normalizar nombre columna fecha → siempre 'date' internamente
date_col_orig = next((c for c in df_raw.columns if c.lower() == 'date'), None)
df_raw = df_raw.rename(columns={date_col_orig: 'date'})
df_raw['date'] = pd.to_datetime(df_raw['date'], errors='coerce')
df_raw = df_raw.sort_values('date').reset_index(drop=True)
 
split_idx = int(len(df_raw) * (1 - TEST_RATIO))
train_raw = df_raw.iloc[:split_idx].copy()
test_raw  = df_raw.iloc[split_idx:].copy()
 
print(f"Total : {len(df_raw)} filas")
print(f"Train : {len(train_raw)} ({train_raw['date'].iloc[0]} → {train_raw['date'].iloc[-1]})")
print(f"Test  : {len(test_raw)}  ({test_raw['date'].iloc[0]}  → {test_raw['date'].iloc[-1]})")
 


## Feature Engineering

In [ ]:

country_holidays = getattr(holidays, {
    "DE": "Germany", "ES": "Spain", "FR": "France",
    "IT": "Italy",   "GB": "UnitedKingdom"
}.get(PAIS, "Germany"))()
 
def add_features(data):
    data = data.copy()
    for lag in range(1, 24):
        data[f'Price_lag_{lag}'] = data['Price'].shift(lag)
    data['rolling_mean'] = data['Price'].rolling(window=3).mean()
    data['rolling_std']  = data['Price'].rolling(window=3).std()
    data['DT'] = data['date'].apply(
        lambda d: 0 if (pd.isna(d) or d.weekday() >= 5 or d in country_holidays) else 1
    )
    data['hour']      = data['date'].dt.hour
    data['dayofweek'] = data['date'].dt.dayofweek
    data['hour_sin']  = np.sin(2 * np.pi * data['hour'] / 24)
    data['hour_cos']  = np.cos(2 * np.pi * data['hour'] / 24)
    data['dow_sin']   = np.sin(2 * np.pi * data['dayofweek'] / 7)
    data['dow_cos']   = np.cos(2 * np.pi * data['dayofweek'] / 7)
    return data
 
train_raw = add_features(train_raw)

## Selection of Features

### Seleccion based on simple correlation. > 0.2

In [ ]:
# set_index before correlation → 'date' IT'S NOT A FEATURE
train_raw = train_raw.dropna()
train_raw = train_raw.set_index('date')
 
corr     = train_raw.corr()['Price'].drop('Price').abs()
selected = ['Price'] + corr[corr >= CORR_THRESHOLD].index.tolist()
 
print(f"\nFeatures seleccionadas: {len(selected)-1} de {len(train_raw.columns)-1}")
print(corr[corr >= CORR_THRESHOLD].sort_values(ascending=False).to_string())
 
train_raw = train_raw[selected]
 
# FINAL_FEATURES defined as a list
FINAL_FEATURES = list(train_raw.columns)
print(f"\nFINAL_FEATURES ({len(FINAL_FEATURES)}): {FINAL_FEATURES}")
 
target_col   = 'Price'
feature_cols = [c for c in FINAL_FEATURES if c != target_col]
 
feature_scaler = MinMaxScaler(feature_range=(0, 1))
target_scaler  = StandardScaler()
 
train_raw[feature_cols] = feature_scaler.fit_transform(train_raw[feature_cols])
train_raw[[target_col]] = target_scaler.fit_transform(train_raw[[target_col]])
 

data_values = train_raw.values
print(f"data_values shape: {data_values.shape}  ← debe coincidir con len(FINAL_FEATURES)={len(FINAL_FEATURES)}")
 

## Secuence creation

In [ ]:
def create_sequences(data, seq_length=SEQ_LENGTH, pred_length=PRED_LENGTH):
    price_idx  = FINAL_FEATURES.index(target_col)  
    sequences, targets = [], []
    for i in range(len(data) - seq_length - pred_length):
        sequences.append(data[i : i + seq_length, :])
        targets.append(data[i + seq_length : i + seq_length + pred_length, price_idx])
    return np.array(sequences), np.array(targets)
 
X_all, y_all = create_sequences(data_values)
X_tensor = torch.tensor(X_all, dtype=torch.float32)
y_tensor = torch.tensor(y_all, dtype=torch.float32)
print(f"X_all shape: {X_all.shape}   y_all shape: {y_all.shape}")

## Split Train / VAL

In [ ]:
val_ratio  = 0.1
val_size   = int(len(X_tensor) * val_ratio)
train_size = len(X_tensor) - val_size
 
train_dataset = TensorDataset(X_tensor[:train_size], y_tensor[:train_size])
val_dataset   = TensorDataset(X_tensor[train_size:], y_tensor[train_size:])
train_loader  = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader    = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False)
 

## Model Structure - LSTM + TRANSFORMER

In [ ]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=500):
        super().__init__()
        pe       = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1).float()
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-np.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe.unsqueeze(0))
 
    def forward(self, x):
        return x + self.pe[:, :x.size(1)]
 
 
class LSTMTransformerHybrid(nn.Module):
    def __init__(self, input_dim,
                 lstm_hidden=64, lstm_num_layers=1,
                 d_model=128, nhead=8, tnum_layers=2,
                 output_length=PRED_LENGTH, dropout_rate=0.1):
        super().__init__()
        self.lstm         = nn.LSTM(input_dim, lstm_hidden,
                                    num_layers=lstm_num_layers, batch_first=True)
        self.dropout1     = nn.Dropout(dropout_rate)
        self.project      = nn.Linear(lstm_hidden, d_model)
        self.pe           = PositionalEncoding(d_model)
        encoder_layer     = nn.TransformerEncoderLayer(
                                d_model=d_model, nhead=nhead,
                                batch_first=True, dropout=dropout_rate)
        self.transformer  = nn.TransformerEncoder(encoder_layer, num_layers=tnum_layers)
        self.dropout2     = nn.Dropout(dropout_rate)
        self.output_layer = nn.Linear(d_model, output_length)
 
    def forward(self, x):
        lstm_out, _ = self.lstm(x)
        lstm_out    = self.dropout1(lstm_out)
        projected   = self.project(lstm_out)
        encoded     = self.pe(projected)
        encoded     = encoded.transpose(0, 1)
        transformed = self.transformer(encoded)
        transformed = transformed.transpose(0, 1)
        final_token = self.dropout2(transformed[:, -1, :])
        return self.output_layer(final_token)
 

## Training

In [ ]:

input_dim = X_all.shape[2]
print(f"input_dim del modelo: {input_dim}  (debe ser {len(FINAL_FEATURES)})")
model     = LSTMTransformerHybrid(input_dim).to(device)
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', patience=5, factor=0.5)
 
best_val_loss     = float('inf')
epochs_no_improve = 0
best_model_state  = None
 
for epoch in range(EPOCHS):
    model.train()
    train_loss = 0.0
    for inputs, targets in train_loader:
        inputs, targets = inputs.to(device), targets.to(device)
        optimizer.zero_grad()
        loss = criterion(model(inputs), targets)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()
 
    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for vi, vt in val_loader:
            vi, vt = vi.to(device), vt.to(device)
            val_loss += criterion(model(vi), vt).item()
 
    train_loss /= len(train_loader)
    val_loss   /= len(val_loader)
    scheduler.step(val_loss)
    print(f"Epoch {epoch+1:3d}/{EPOCHS} | Train: {train_loss:.4f} | Val: {val_loss:.4f}")
 
    if val_loss < best_val_loss:
        best_val_loss     = val_loss
        best_model_state  = model.state_dict()
        epochs_no_improve = 0
    else:
        epochs_no_improve += 1
        if epochs_no_improve >= PATIENCE_ES:
            print(f"Early stopping en época {epoch+1}")
            break

# Save best model
model.load_state_dict(best_model_state)
torch.save(best_model_state, "best_model.pt")
print("Modelo guardado en best_model.pt")
 

## Pre processing for testing

In [ ]:

test_raw = test_raw.copy()

# Date normalization
dc = next((c for c in test_raw.columns if c.lower() == 'date'), None)
if dc and dc != 'date':
    test_raw = test_raw.rename(columns={dc: 'date'})
test_raw['date'] = pd.to_datetime(test_raw['date'], errors='coerce')
 
# real prices set up vs prediction
test_prices       = test_raw['Price'].values
last_train_prices = df_raw.iloc[:split_idx]['Price'].values[-23:]
combined_prices   = np.concatenate([last_train_prices, test_prices])
 
for lag in range(1, 24):
    test_raw[f'Price_lag_{lag}'] = combined_prices[-(len(test_raw) + lag):-lag]
 
# Rolling con continuidad desde training
rolling_seed   = df_raw.iloc[:split_idx]['Price'].values[-2:]
rolling_window = np.concatenate([rolling_seed, test_prices])
test_raw['rolling_mean'] = [rolling_window[i:i+3].mean() for i in range(len(test_prices))]
test_raw['rolling_std']  = [rolling_window[i:i+3].std()  for i in range(len(test_prices))]
test_raw[['rolling_mean','rolling_std']] = test_raw[['rolling_mean','rolling_std']].ffill()
 
# Calendar features
test_raw['DT'] = test_raw['date'].apply(
    lambda d: 0 if (pd.isna(d) or d.weekday() >= 5 or d in country_holidays) else 1
)
test_raw['hour']      = test_raw['date'].dt.hour
test_raw['dayofweek'] = test_raw['date'].dt.dayofweek
test_raw['hour_sin']  = np.sin(2 * np.pi * test_raw['hour'] / 24)
test_raw['hour_cos']  = np.cos(2 * np.pi * test_raw['hour'] / 24)
test_raw['dow_sin']   = np.sin(2 * np.pi * test_raw['dayofweek'] / 7)
test_raw['dow_cos']   = np.cos(2 * np.pi * test_raw['dayofweek'] / 7)
 
# VMD 
train_prices_raw = df_raw.iloc[:split_idx]['Price'].dropna().values
full_prices      = np.concatenate([train_prices_raw, test_prices])
test_start_idx   = len(train_prices_raw)
window_size, K   = 750, 3
vmd_components   = [np.full(len(test_raw), np.nan) for _ in range(K)]
 
for day in tqdm(range(len(test_raw) // 24), desc="VMD Test"):
    end_idx   = test_start_idx + day * 24 + 24
    start_idx = max(0, end_idx - window_size)
    modes, _, _ = VMD(full_prices[start_idx:end_idx],
                      alpha=2000, tau=0, K=K, DC=0, init=1, tol=1e-7)
    for k in range(K):
        vmd_components[k][day * 24 : day * 24 + 24] = modes[k][-24:]
 
for i in range(K):
    test_raw[f'VMD_LogComp_{i+1}'] = vmd_components[i]
 
test_raw = test_raw.dropna()
test_raw = test_raw.set_index('date')
 
# Selection of exact model columns
missing = [c for c in FINAL_FEATURES if c not in test_raw.columns]
if missing:
    raise ValueError(f"❌ Columnas faltantes en test: {missing}")
 
test_raw = test_raw[FINAL_FEATURES]  # orden y columnas exactas
 
# Scaling con scalers del training
feat_cols_test          = [c for c in FINAL_FEATURES if c != target_col]
test_raw[feat_cols_test] = feature_scaler.transform(test_raw[feat_cols_test])
test_raw[[target_col]]   = target_scaler.transform(test_raw[[target_col]])
 
test_dat_vals = test_raw.values
 
print(f"data_values shape   : {data_values.shape}")
print(f"test_dat_vals shape : {test_dat_vals.shape}")


## Model Evaluation

In [ ]:

model.eval()
 
actual_prices_full = df_raw.iloc[split_idx:]['Price'].values
start_date         = pd.to_datetime(df_raw.iloc[split_idx]['date'])
combined_data      = np.vstack([data_values, test_dat_vals])
 
all_rmses, all_r2s, all_maes, all_mapes = [], [], [], []
results = []
 
for day in range(DAYS_PREDICT):
    seq_start = len(data_values) - SEQ_LENGTH + day * PRED_LENGTH
    seq_end   = seq_start + SEQ_LENGTH
 
    if seq_end > len(combined_data):
        continue
 
    input_tensor = torch.tensor(
        combined_data[seq_start:seq_end], dtype=torch.float32
    ).unsqueeze(0).to(device)
 
    with torch.no_grad():
        forecast = model(input_tensor).cpu().numpy().flatten()
 
    start_gt, end_gt = day * PRED_LENGTH, (day + 1) * PRED_LENGTH
    if end_gt > len(actual_prices_full):
        continue
 
    actual_prices     = actual_prices_full[start_gt:end_gt]
    forecast_unscaled = target_scaler.inverse_transform(
        forecast.reshape(-1, 1)
    ).flatten()
 
    rmse = np.sqrt(mean_squared_error(actual_prices, forecast_unscaled))
    r2   = r2_score(actual_prices, forecast_unscaled)
    mae  = mean_absolute_error(actual_prices, forecast_unscaled)
    mask = actual_prices > EPSILON_MAPE
    mape = (np.mean(np.abs(
        (actual_prices[mask] - forecast_unscaled[mask]) / actual_prices[mask]
    )) * 100) if np.any(mask) else np.nan
 
    print(f"Día {day+1:2d} | RMSE: {rmse:.2f} | R²: {r2:.4f} | MAE: {mae:.2f} | MAPE: {mape:.2f}%")
    all_rmses.append(rmse); all_r2s.append(r2)
    all_maes.append(mae);   all_mapes.append(mape)
 
    ci_margin      = 1.96 * np.std(actual_prices - forecast_unscaled)
    forecast_dates = [start_date + timedelta(hours=start_gt + h) for h in range(PRED_LENGTH)]
 
    for h in range(PRED_LENGTH):
        pred = forecast_unscaled[h]
        results.append({
            "Day": day + 1, "Hour": h,
            "Date": forecast_dates[h],
            "Actual Price":    actual_prices[h],
            "Predicted Price": pred,
            "Lower CI":        pred - ci_margin,
            "Upper CI":        pred + ci_margin,
            "RMSE": rmse, "R²": r2, "MAE": mae, "MAPE": mape
        })
 
print(f"\n{'='*50}")
print(f"RMSE medio  : {np.mean(all_rmses):.4f}")
print(f"MAE medio   : {np.mean(all_maes):.4f}")
print(f"MAPE medio  : {np.nanmean(all_mapes):.2f}%")
print(f"R² medio    : {np.mean(all_r2s):.4f}")
 
results_df = pd.DataFrame(results)

In [ ]:
common_cols = test_raw.columns.intersection(train_raw.columns)

train_raw = train_raw[common_cols]
test_raw  = test_raw[common_cols]

data_values   = train_raw.values
test_dat_vals = test_raw.values

## Evaluation TEST SET

In [ ]:
model.eval()


actual_prices_full = df_raw.iloc[split_idx:]['Price'].values
start_date         = df_raw.iloc[split_idx]['date']
combined_data = np.vstack([data_values, test_dat_vals])

all_rmses, all_r2s, all_maes, all_mapes = [], [], [], []
results = []
 
for day in range(DAYS_PREDICT):
    seq_start = len(data_values) - SEQ_LENGTH + day * PRED_LENGTH
    seq_end   = seq_start + SEQ_LENGTH
 
    if seq_end > len(combined_data):
        continue
 
    input_tensor = torch.tensor(
        combined_data[seq_start:seq_end], dtype=torch.float32
    ).unsqueeze(0).to(device)
 
    with torch.no_grad():
        forecast = model(input_tensor).cpu().numpy().flatten()
 
    start_gt, end_gt = day * PRED_LENGTH, (day + 1) * PRED_LENGTH
    if end_gt > len(actual_prices_full):
        continue
 
    actual_prices     = actual_prices_full[start_gt:end_gt]
    forecast_unscaled = target_scaler.inverse_transform(
        forecast.reshape(-1, 1)
    ).flatten()
 
    rmse = np.sqrt(mean_squared_error(actual_prices, forecast_unscaled))
    r2   = r2_score(actual_prices, forecast_unscaled)
    mae  = mean_absolute_error(actual_prices, forecast_unscaled)
    mask = actual_prices > EPSILON_MAPE
    mape = (np.mean(np.abs(
        (actual_prices[mask] - forecast_unscaled[mask]) / actual_prices[mask]
    )) * 100) if np.any(mask) else np.nan
 
    print(f"Day {day+1:2d} | RMSE: {rmse:.2f} | R²: {r2:.4f} | MAE: {mae:.2f} | MAPE: {mape:.2f}%")
    all_rmses.append(rmse); all_r2s.append(r2)
    all_maes.append(mae);   all_mapes.append(mape)
 
    ci_margin      = 1.96 * np.std(actual_prices - forecast_unscaled)
    forecast_dates = [start_date + timedelta(hours=start_gt + h) for h in range(PRED_LENGTH)]
 
    for h in range(PRED_LENGTH):
        pred = forecast_unscaled[h]
        results.append({
            "Day": day + 1, "Hour": h,
            "Date": forecast_dates[h],
            "Actual Price": actual_prices[h],
            "Predicted Price": pred,
            "Lower CI": pred - ci_margin,
            "Upper CI": pred + ci_margin,
            "RMSE": rmse, "R²": r2, "MAE": mae, "MAPE": mape
        })
 
print(f"\n{'='*50}")
print(f"RMSE   : {np.mean(all_rmses):.4f}")
print(f"MAE    : {np.mean(all_maes):.4f}")
print(f"MAPEo  : {np.nanmean(all_mapes):.2f}%")
print(f"R²     : {np.mean(all_r2s):.4f}")
 
results_df = pd.DataFrame(results)

## VISUALIZATION

In [ ]:
# --- Gráfico por día ---
nrows = (DAYS_PREDICT + 1) // 2
fig, axs = plt.subplots(nrows=nrows, ncols=2, figsize=(15, nrows * 3))
axs = axs.flatten()

for day in range(1, DAYS_PREDICT + 1):
    subset = results_df[results_df["Day"] == day]
    ax = axs[day - 1]
    ax.plot(subset["Date"], subset["Actual Price"],    label="Real",       color='black')
    ax.plot(subset["Date"], subset["Predicted Price"], label="Predicción", color='blue', linestyle='--')
    ax.fill_between(subset["Date"], subset["Lower CI"], subset["Upper CI"],
                    color='blue', alpha=0.2, label='IC 95%')
    ax.set_title(
        f"Day {day} | RMSE:{subset['RMSE'].iloc[0]:.2f} R²:{subset['R²'].iloc[0]:.2f} "
        f"MAE:{subset['MAE'].iloc[0]:.2f} MAPE:{subset['MAPE'].iloc[0]:.2f}%", fontsize=9
    )
    ax.set_ylabel("Price (EUR/MWh)")
    ax.legend(fontsize=8)

for i in range(DAYS_PREDICT, len(axs)):
    axs[i].set_visible(False)

plt.tight_layout()
plt.suptitle("LSTM+Transformer", fontsize=14, y=1.01)
plt.savefig("daily_forecast.png", dpi=150, bbox_inches='tight')
plt.show()

# --- Gráfico continuo ---
results_df['Date'] = pd.to_datetime(results_df['Date'])
unique_days        = results_df['Date'].dt.normalize().unique()

fig, ax = plt.subplots(figsize=(23 / 2.54, 15 / 2.54))
ax.plot(results_df["Date"], results_df["Actual Price"],    label="Real",       color='black')
ax.plot(results_df["Date"], results_df["Predicted Price"], label="Prediction", color='blue', linestyle='--')
ax.fill_between(results_df["Date"], results_df["Lower CI"], results_df["Upper CI"],
                color='#0091ff', alpha=0.2, label='IC 95%')

for day_start in unique_days:
    ax.axvline(x=day_start, color='gray', linestyle=':', alpha=0.7)
    ax.text(day_start + timedelta(hours=12),
            ax.get_ylim()[1] * 0.95,
            pd.to_datetime(day_start).strftime('%A'),
            ha='center', fontsize=8, color='gray')

ax.set_ylabel("Price (EUR/MWh)")
ax.legend()
ax.grid(True, linestyle='--', alpha=0.4)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m-%d %H:%M'))
plt.xticks(rotation=25)
plt.tight_layout()
plt.savefig(OUTPUT_SVG, format="svg", bbox_inches='tight', transparent=True)
plt.show()
print(f"Gráfico exportado → {OUTPUT_SVG}")